In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import copy
import sys
import random
import torchvision
import torchvision.transforms as transforms
import torch
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)
from src.trainer import IntervalTrainer, InterContiNetTrainer

# Define transform (e.g., convert to tensor)
transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5] * 3, std=[0.5] * 3),
    ]
)

from src.data_utils import split_mnist_by_labels, get_context_sets
from src.utils import general as utils
import src.models as models

import pandas as pd
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import json

from src.utils.plotting_style import set_figure_size
from src.regulariser import L2Regulariser, UnbiasRegulariser, MultiRegulariser

In [3]:
device = "cuda"
classes = [
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck",
]

task_pairs = [
    ("cat", "truck"),
    ("frog", "ship"),
    ("horse", "automobile"),
    ("dog", "airplane"),
    ("bird", "deer"),
]
task_pairs_ids = [(classes.index(i), classes.index(j)) for i, j in task_pairs]
animals_mask = torch.tensor([0, 0, 1, 1, 1, 1, 1, 1, 0, 0]).to(device)


def domain_map_fn(y):
    return animals_mask[y]


@torch.no_grad()
def encode(x, model, device="cuda"):
    # Handle batching to avoid out-of-memory issues
    batch_size = 2048
    features = []
    for i in range(0, len(x), batch_size):
        batch = x[i : i + batch_size].to(device)
        batch_features = model(batch).flatten(start_dim=1).cpu()
        features.append(batch_features)
    return torch.cat(features, dim=0)


def encode_dataset(train_dataset, test_dataset, encoder, device="cuda"):
    train_imgs, train_labels = train_dataset.data, train_dataset.targets
    test_imgs, test_labels = test_dataset.data, test_dataset.targets
    # apply the appropriate scaling and transposition
    train_imgs = (
        torch.tensor(train_imgs, dtype=torch.float32).permute(0, 3, 1, 2) / 255.0
    )
    test_imgs = torch.tensor(test_imgs, dtype=torch.float32).permute(0, 3, 1, 2) / 255.0
    train_imgs = (train_imgs - 0.5) / 0.5
    test_imgs = (test_imgs - 0.5) / 0.5
    train_labels = torch.tensor(train_labels, dtype=torch.int64).squeeze()
    test_labels = torch.tensor(test_labels, dtype=torch.int64).squeeze()

    if encoder is not None:
        train_imgs = encode(train_imgs, encoder, device)
        test_imgs = encode(test_imgs, encoder, device)

    train_dataset = torch.utils.data.TensorDataset(train_imgs, train_labels)
    test_dataset = torch.utils.data.TensorDataset(test_imgs, test_labels)
    return train_dataset, test_dataset


def get_tasks(encoder):
    non_animals = [0, 1, 8, 9]
    animals = [2, 3, 4, 5, 6, 7]

    non_animal_indices = torch.tensor(non_animals)[torch.randperm(4)].tensor_split(5)
    animal_indices = list(torch.tensor(animals)[torch.randperm(6)].tensor_split(5))
    animal_indices.reverse()

    task_pairs_ids = [
        t.tolist() + n.tolist() for t, n in zip(animal_indices, non_animal_indices)
    ]
    print("Contexts:", task_pairs_ids)
    trainset = torchvision.datasets.CIFAR10(
        root="./data", train=True, download=True, transform=transform
    )
    testset = torchvision.datasets.CIFAR10(
        root="./data", train=False, download=True, transform=transform
    )
    trainset.targets = torch.tensor(trainset.targets)
    testset.targets = torch.tensor(testset.targets)

    if encoder is not None:
        trainset, testset = encode_dataset(trainset, testset, encoder)
    test_tasks = [
        split_mnist_by_labels(testset, task_pair_id) for task_pair_id in task_pairs_ids
    ]
    train_tasks = [
        split_mnist_by_labels(trainset, task_pair_id) for task_pair_id in task_pairs_ids
    ]

    return train_tasks, test_tasks

In [4]:
# Get the CIFAR 100 dataset
cifar100_trainset = torchvision.datasets.CIFAR100(
    root="./data", train=True, download=True, transform=transform
)
cifar100_testset = torchvision.datasets.CIFAR100(
    root="./data", train=False, download=True, transform=transform
)

# Convert targets to tensor for consistency
cifar100_trainset.targets = torch.tensor(cifar100_trainset.targets)
cifar100_testset.targets = torch.tensor(cifar100_testset.targets)

# Print dataset info
print(f"CIFAR-100 training set: {len(cifar100_trainset)} images")
print(f"CIFAR-100 test set: {len(cifar100_testset)} images")
print(f"Number of classes: {len(set(cifar100_trainset.targets.tolist()))}")

# Sample a few class names
print(f"Sample classes: {cifar100_trainset.classes[:10]}")

CIFAR-100 training set: 50000 images
CIFAR-100 test set: 10000 images
Number of classes: 100
Sample classes: ['apple', 'aquarium_fish', 'baby', 'bear', 'beaver', 'bed', 'bee', 'beetle', 'bicycle', 'bottle']


In [5]:
# encoder = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT).to(device)
# encoder.fc = torch.nn.Linear(512, 100)
# encoder.to(device)
# # task0_train, task0_test = get_tasks(None)[0]
# pretrain_data_train, pretrain_data_test = cifar100_trainset, cifar100_testset
# simple_trainer = SimpleTrainer(encoder)
# simple_trainer.train(pretrain_data_train, pretrain_data_test, n_epochs=4, learning_rate=0.001, val_freq=200)
# base_model = encoder
# # Create a directory for saved models if it doesn't exist
# save_dir = os.path.join(project_root, "notebooks/saved_models")
# os.makedirs(save_dir, exist_ok=True)

# # Save the complete model
# model_path = os.path.join(save_dir, "resnet18_cifar100_pretrained.pth")
# torch.save(base_model.state_dict(), model_path)

In [6]:
# Create a simple function to load the model
save_dir = os.path.join(project_root, "notebooks/saved_models")
model_path = os.path.join(save_dir, "resnet18_cifar100_pretrained.pth")


def load_pretrained_model(model_path, num_classes=100):
    model = torchvision.models.resnet18(weights=None)
    model.fc = torch.nn.Linear(512, num_classes)
    model.load_state_dict(torch.load(model_path))
    return model


model = load_pretrained_model(model_path)

In [7]:
encoder = copy.deepcopy(model).cuda()
encoder.fc = torch.nn.Identity()

In [8]:
X_min, X_max = None, None
for task in get_tasks(encoder):
    X, _ = next(
        iter(torch.utils.data.DataLoader(task[0], batch_size=10000, shuffle=False))
    )
    if X_min is None:
        X_min, X_max = X.min(dim=0).values, X.max(dim=0).values
    else:
        X_min = torch.min(X_min, X.min(dim=0).values)
        X_max = torch.max(X_max, X.max(dim=0).values)
X_min, X_max = X_min.to(device), X_max.to(device)

Contexts: [[6, 8], [7, 0], [3, 9], [4, 1], [2, 5]]


/tmp/ipykernel_1382905/1168962829.py:52: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  train_labels = torch.tensor(train_labels, dtype=torch.int64).squeeze()
/tmp/ipykernel_1382905/1168962829.py:53: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_labels = torch.tensor(test_labels, dtype=torch.int64).squeeze()


In [9]:
from collections import defaultdict
from src.helpers.WandbWrapper import WandbTrainerWrapper
from src.models import get_mnist_model
from src.data_utils import get_mnist_tasks
from src.utils.general import set_seed

import wandb

results = {
    "class": defaultdict(list),
    "task": defaultdict(list),
    "domain": defaultdict(list),
}

In [13]:
def run_cil():
    config = {
        "ours": True,
        "init.projection_strategy": "best_loss",
        "init.n_certificate_samples": 400,
        "init.min_acc_limit": 1,
        "init.min_acc_increment": 0.2,
        "init.paradigm": "CIL",
        "init.n_iters": 200,
        "init.primal_learning_rate": 0.33,
        "init.dual_learning_rate": 0.01,
        "init.penalty_coefficient": 1,
        "init.checkpoint": 2,
        "train.l2_lambda": 0.01,
        "train.unbias_lambda": 0.01,
        "train.lr": 0.02,
        "train.weight_decay": 0,
        "train.epochs": 5,
        "train.batch_size": 128,
        "simple_train.epochs": 5,
        "simple_train.batch_size": 128,
        "simple_train.lr": 0.02,
        "simple_train.weight_decay": 0,
        "benchmarks": {
            "ewc": {
                "lmbd": 1e6,
                "fisher_batch": 64,
                "lr": 0.02,
                "weight_decay": 0,
                "epochs": 5,
                "batch_size": 128,
            },
            "sgd": {
                "lr": 0.02,
                "weight_decay": 0,
                "epochs": 5,
                "batch_size": 128,
            },
            "lwf": {
                "lmbd": 0.1,
                "temp": 2,
                "lr": 0.02,
                "weight_decay": 0,
                "epochs": 5,
                "batch_size": 128,
            },
            "icn": {
                "lr": 0.01,
                "batch_size": 128,
                "epochs": 5,
                "lid_lr": 0.1,
                "min_acc_limit": 1,
                "min_acc_increment": 0.2,
            }
        },
    }
    tag = "final_cifar_cil_new"
    benchmark_tags = [
        f"final_cifar_cil_{bench}" for bench in config["benchmarks"].keys()
    ]

    for i in range(15):
        set_seed(i)
        config["init.seed"] = i
        train_tasks, test_tasks = get_tasks(encoder)
        model = models.get_fully_connected_model(
            input_dim=512, seed=config["init.seed"], device="cuda", output_dim=10
        )
        wrapper = WandbTrainerWrapper(
            trainer_class=IntervalTrainer,
            model=model,
            train_tasks=train_tasks,
            val_tasks=test_tasks,
            test_tasks=test_tasks,
            input_dim=512,
            seed=i
        )
        wrapper.run(
            project="certified-continual-learning",
            tags=["final_cifar_new"]
            + ([tag] if config["ours"] else [])
            + benchmark_tags,
            config=config,
            unbias_domain=[X_min, X_max],
        )


run_cil()


Contexts: [[4, 1], [6, 0], [7, 8], [2, 9], [3, 5]]


/tmp/ipykernel_1382905/1168962829.py:52: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  train_labels = torch.tensor(train_labels, dtype=torch.int64).squeeze()
/tmp/ipykernel_1382905/1168962829.py:53: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_labels = torch.tensor(test_labels, dtype=torch.int64).squeeze()


Training Epochs: 100%|███████████████████████████████████████████████| 5/5 [00:02<00:00,  2.26it/s, val_loss=0.0000, val_acc=None, proj=None, progress=0.99]


Test Results: [(0.4614, 0.8575), (26.9735, 0.0), (28.4993, 0.0), (41.6077, 0.0), (33.1331, 0.0)] (Avg: (26.1350, 0.1715))
Initial acc constraint violation: -0.1756 (Positive = violated)
[tensor(0.6675, device='cuda:0'), tensor(0.6675, device='cuda:0'), tensor(0.6675, device='cuda:0'), tensor(0.6675, device='cuda:0'), tensor(0.6675, device='cuda:0')]
Number of model parameters: 5130
Computing Rashomon set with min acc limit: 0.67
Initial bbox:  Obj=0.00,  Size=0.00,  Min acc hard=0.84,  Min acc soft=0.84


100%|████████████████████████████████████████████████████████████████████████| 200/200 [00:15<00:00, 13.21it/s, size=1235.03, obj=0.240, min_soft_acc=0.687]


...

Training Epochs: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:01<00:00,  3.04it/s]


Test Results: [(0.9038, 0.718), (2.7642, 0.0365), (2.4584, 0.089), (2.76, 0.0605), (108.899, 0.0)] (Avg: (23.5571, 0.1808))
Failed to set context: Last model layer is not a context head.


Training Epochs: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:01<00:00,  2.77it/s]


Test Results: [(1.2293, 0.5725), (2.1648, 0.2185), (2.7841, 0.0535), (3.9327, 0.004), (2.7176, 0.052)] (Avg: (2.5657, 0.1801))


final_avg_accuracy,▁
final_avg_loss,▁
final_num_tasks,▁
final_total_accuracy,▁
second_task_accuracy,▁
seed,▁
final_avg_accuracy,0.1903
final_avg_loss,2.38172
final_num_tasks,5
final_total_accuracy,0.9515
second_task_accuracy,0.1885


In [12]:
def run_dil():
    animals_mask = torch.tensor([0, 0, 1, 1, 1, 1, 1, 1, 0, 0]).to(device)

    def domain_map_fn(y):
        return animals_mask[y]

    config = {
        "ours": True,
        "init.projection_strategy": "best_loss",
        "init.n_certificate_samples": 400,
        "init.min_acc_limit": 1,
        "init.min_acc_increment": 0.2,
        "init.paradigm": "DIL",
        "init.n_iters": 200,
        "init.primal_learning_rate": 0.33,
        "init.dual_learning_rate": 0.01,
        "init.penalty_coefficient": 1,
        "init.checkpoint": 2,
        "train.l2_lambda": 0.01,
        "train.unbias_lambda": 0.01,
        "train.lr": 0.02,
        "train.weight_decay": 0,
        "train.epochs": 5,
        "train.batch_size": 128,
        "simple_train.epochs": 5,
        "simple_train.batch_size": 128,
        "simple_train.lr": 0.02,
        "simple_train.weight_decay": 0,
        "benchmarks": {
            "ewc": {
                "lmbd": 1e6,
                "fisher_batch": 64,
                "lr": 0.02,
                "weight_decay": 0,
                "epochs": 5,
                "batch_size": 128,
            },
            "sgd": {
                "lr": 0.02,
                "weight_decay": 0,
                "epochs": 5,
                "batch_size": 128,
            },
            "lwf": {
                "lmbd": 0.1,
                "temp": 2,
                "lr": 0.02,
                "weight_decay": 0,
                "epochs": 5,
                "batch_size": 128,
            },
            "icn": {
                "lr": 0.01,
                "batch_size": 128,
                "epochs": 30,
                "lid_lr": 1,
                "min_acc_limit": 1,
                "min_acc_increment": 0.2,
            }
        },
    }
    tag = "final_cifar_dil_new"
    benchmark_tags = [
        f"final_cifar_dil_{bench}" for bench in config["benchmarks"].keys()
    ]

    for i in range(10, 15):
        set_seed(i)
        config["init.seed"] = i
        train_tasks, test_tasks = get_tasks(encoder)
        model = models.get_fully_connected_model(
            input_dim=512, seed=config["init.seed"], device="cuda", output_dim=2
        )
        wrapper = WandbTrainerWrapper(
            trainer_class=IntervalTrainer,
            model=model,
            train_tasks=train_tasks,
            val_tasks=test_tasks,
            test_tasks=test_tasks,
            domain_map_fn=domain_map_fn,
            input_dim=512,
            seed=i,
        )
        wrapper.run(
            project="certified-continual-learning",
            tags=["final_cifar_new"]
            + ([tag] if config["ours"] else [])
            + benchmark_tags,
            config=config,
            unbias_domain=[X_min, X_max],
        )


run_dil()

Contexts: [[4, 1], [6, 0], [7, 8], [2, 9], [3, 5]]


/tmp/ipykernel_1382905/1168962829.py:52: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  train_labels = torch.tensor(train_labels, dtype=torch.int64).squeeze()
/tmp/ipykernel_1382905/1168962829.py:53: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_labels = torch.tensor(test_labels, dtype=torch.int64).squeeze()


Training Epochs: 100%|███████████████████████████████████████████████| 5/5 [00:02<00:00,  2.37it/s, val_loss=0.0000, val_acc=None, proj=None, progress=0.99]


Test Results: [(0.5494, 0.8425), (1.7234, 0.6085), (1.5101, 0.638), (1.5105, 0.7425), (3.6576, 0.4245)] (Avg: (1.7902, 0.6512))
Initial acc constraint violation: -0.2169 (Positive = violated)
[tensor(0.6250, device='cuda:0'), tensor(0.6250, device='cuda:0'), tensor(0.6250, device='cuda:0'), tensor(0.6250, device='cuda:0'), tensor(0.6250, device='cuda:0')]
Number of model parameters: 1026
Computing Rashomon set with min acc limit: 0.62
Initial bbox:  Obj=0.00,  Size=0.00,  Min acc hard=0.84,  Min acc soft=0.84


100%|██████████████████████████████████████████████████████████████████████████| 200/200 [00:11<00:00, 17.06it/s, size=19.87, obj=0.019, min_soft_acc=0.610]


...

Training Epochs: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:01<00:00,  2.58it/s]


Test Results: [(0.3649, 0.8825), (0.5963, 0.789), (1.1402, 0.6405), (1.7796, 0.5345), (0.953, 0.7)] (Avg: (0.9668, 0.7093))


Training Epochs: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:02<00:00,  2.03it/s]


Test Results: [(0.3645, 0.8825), (0.5958, 0.788), (1.1396, 0.641), (1.7805, 0.535), (0.9473, 0.7025)] (Avg: (0.9655, 0.7098))


final_avg_accuracy,▁
final_avg_loss,▁
final_num_tasks,▁
final_total_accuracy,▁
second_task_accuracy,▁
seed,▁
final_avg_accuracy,0.7147
final_avg_loss,1.36258
final_num_tasks,5
final_total_accuracy,3.5735
second_task_accuracy,0.7525


In [11]:
def run_til():
    config = {
        "ours": True,
        "init.projection_strategy": "best_loss",
        "init.n_certificate_samples": 400,
        "init.min_acc_limit": 1,
        "init.min_acc_increment": 0.2,
        "init.paradigm": "TIL",
        "init.n_iters": 200,
        "init.primal_learning_rate": 0.33,
        "init.dual_learning_rate": 0.01,
        "init.penalty_coefficient": 1,
        "init.checkpoint": 2,
        "train.l2_lambda": 0.01,
        "train.unbias_lambda": 0.01,
        "train.lr": 0.02,
        "train.weight_decay": 0,
        "train.epochs": 5,
        "train.batch_size": 128,
        "simple_train.epochs": 5,
        "simple_train.batch_size": 128,
        "simple_train.lr": 0.02,
        "simple_train.weight_decay": 0,
        "benchmarks": {
            "ewc": {
                "lmbd": 1e6,
                "fisher_batch": 64,
                "lr": 0.02,
                "weight_decay": 0,
                "epochs": 5,
                "batch_size": 128,
            },
            "sgd": {
                "lr": 0.02,
                "weight_decay": 0,
                "epochs": 5,
                "batch_size": 128,
            },
            "lwf": {
                "lmbd": 0.1,
                "temp": 2,
                "lr": 0.02,
                "weight_decay": 0,
                "epochs": 5,
                "batch_size": 128,
            },
            "icn": {
                "lr": 0.01,
                "batch_size": 128,
                "epochs": 30,
                "lid_lr": 1,
                "min_acc_limit": 1,
                "min_acc_increment": 0.2,
            }
        },
    }
    tag = "final_cifar_til_new"
    benchmark_tags = [
        f"final_cifar_til_{bench}" for bench in config["benchmarks"].keys()
    ]

    for i in range(10, 15):
        set_seed(i)
        config["init.seed"] = i
        train_tasks, test_tasks = get_tasks(encoder)
        context_sets = get_context_sets(test_tasks)
        head = utils.InContextHead(context_sets, context_dim=10, device=device)
        model = models.get_fully_connected_model(
            input_dim=512,
            seed=config["init.seed"],
            device="cuda",
            output_dim=10,
            head=head,
        )
        wrapper = WandbTrainerWrapper(
            trainer_class=IntervalTrainer,
            model=model,
            train_tasks=train_tasks,
            val_tasks=test_tasks,
            test_tasks=test_tasks,
            input_dim=512,
            seed=i
        )
        wrapper.run(
            project="certified-continual-learning",
            tags=["final_cifar_new"]
            + ([tag] if config["ours"] else [])
            + benchmark_tags,
            config=config,
            unbias_domain=[X_min, X_max],
        )


run_til()

Contexts: [[4, 1], [6, 0], [7, 8], [2, 9], [3, 5]]


/tmp/ipykernel_1382905/1168962829.py:52: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  train_labels = torch.tensor(train_labels, dtype=torch.int64).squeeze()
/tmp/ipykernel_1382905/1168962829.py:53: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_labels = torch.tensor(test_labels, dtype=torch.int64).squeeze()
wandb: Currently logged in as: le24 (agt-team) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Training Epochs: 100%|███████████████████████████████████████████████| 5/5 [00:03<00:00,  1.36it/s, val_loss=0.0000, val_acc=None, proj=None, progress=0.99]


Test Results: [(0.3272, 0.8895), (2.3026, 0.453), (2.3026, 0.2785), (2.3026, 0.359), (2.3026, 0.381)] (Avg: (1.9075, 0.4722))
Initial acc constraint violation: -0.1697 (Positive = violated)
[tensor(0.7150, device='cuda:0'), tensor(0.7150, device='cuda:0'), tensor(0.7150, device='cuda:0'), tensor(0.7150, device='cuda:0'), tensor(0.7150, device='cuda:0')]
Number of model parameters: 5130
Computing Rashomon set with min acc limit: 0.71
Initial bbox:  Obj=16.00,  Size=82080.00,  Min acc hard=0.88,  Min acc soft=0.88


100%|██████████████████████████████████████████████████████████████████████| 200/200 [00:15<00:00, 12.74it/s, size=82208.38, obj=16.025, min_soft_acc=0.784]


...

Training Epochs: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:03<00:00,  1.37it/s]


Test Results: [(0.4971, 0.7715), (1.4665, 0.775), (1.9913, 0.6215), (1.8562, 0.704), (2.4612, 0.3705)] (Avg: (1.6545, 0.6485))


Training Epochs: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:03<00:00,  1.26it/s]


Test Results: [(0.4971, 0.7715), (1.4665, 0.775), (1.9913, 0.6215), (1.8562, 0.704), (2.161, 0.5885)] (Avg: (1.5944, 0.6921))


final_avg_accuracy,▁
final_avg_loss,▁
final_num_tasks,▁
final_total_accuracy,▁
second_task_accuracy,▁
seed,▁
final_avg_accuracy,0.8422
final_avg_loss,0.45432
final_num_tasks,5
final_total_accuracy,4.211
second_task_accuracy,0.891
